# Opis projektu i dobór priorów

Ten notebook opisuje aktualną wersję projektu: **modele drużynowe** (Stan), które przewidują punkty **jednej drużyny** w sezonie. Złożenie pełnej tabeli ligowej z wielu takich predykcji jest pokazane osobno w notebooku **05** (`05_backtest_models_comparison`).

W projekcie używamy dwóch modeli drużynowych:

- `stan/team_static.stan` - model statyczny z jednym globalnym skillem drużyny,
- `stan/team_hierarchical.stan` - model hierarchiczny z trwałą jakością drużyny i sezonowym odchyleniem.

Oba modele mają **wejście = jedna drużyna** (cechy procesowe + indeks klubu) i **wyjście = rozkład punktów** tej drużyny w sezonie.

### Notebooki w projekcie

| Plik | Treść |
|------|--------|
| `00_project_overview_and_priors` | Ten notebook — opis projektu i priory |
| `01_eda_and_tables` | EDA i budowa tabel sezonowych |
| `02_model1_static_team` | Model 1 (static), fit, PPC, backtest |
| `03_model2_hierarchical_team` | Model 2 (hierarchical), fit, PPC, backtest |
| `04_forecast_2627_comparison` | Forecast 2026/27, LOO/WAIC, porównanie modeli |
| `05_backtest_models_comparison` | Backtest 2025/26 — Model 1 vs Model 2 |

In [1]:
import numpy as np
import pandas as pd

from helping_functions import (
    load_matches,
    load_season_tables,
    prepare_table_stan_static,
    prepare_table_stan_hierarchical,
    BACKTEST_TRAIN_SEASONS,
    BACKTEST_TEST_SEASON,
    FORECAST_TRAIN_SEASONS,
    FORECAST_SEASON_LABEL,
    STUDENT_T_NU,
)

matches = load_matches()
tables = load_season_tables(matches, BACKTEST_TRAIN_SEASONS)

print(f"Mecze w danych: {len(matches)}")
print(f"Sezony w danych: {sorted(matches['season'].unique())}")
print(f"Sezony treningowe backtestu: {BACKTEST_TRAIN_SEASONS[0]}-{BACKTEST_TRAIN_SEASONS[-1]}")
print(f"Sezon testowy backtestu: {BACKTEST_TEST_SEASON}")
print(f"Sezony treningowe finalnego forecastu: {FORECAST_TRAIN_SEASONS[0]}-{FORECAST_TRAIN_SEASONS[-1]}")
print(f"Forecast sezonu: {FORECAST_SEASON_LABEL}")
print(f"Student-t nu w modelach drużynowych: {STUDENT_T_NU}")

Mecze w danych: 6460
Sezony w danych: ['0910', '1011', '1112', '1213', '1314', '1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425', '2526']
Sezony treningowe backtestu: 0910-2425
Sezon testowy backtestu: 2526
Sezony treningowe finalnego forecastu: 0910-2526
Forecast sezonu: 2627
Student-t nu w modelach drużynowych: 5.0


## Cel projektu

Celem jest probabilistyczne przewidywanie **liczby punktów drużyny** w sezonie Premier League. Modele nie przewidują pojedynczych wyników meczów ani całej tabeli naraz — każda predykcja dotyczy **jednego klubu**.

Przepływ jest następujący:

1. Wczytujemy historyczne mecze.
2. Z meczów budujemy końcowe tabele sezonów (dane treningowe).
3. Dla każdej pary drużyna–sezon wyliczamy cechy opisujące jakość i formę.
4. Trenujemy model Stan na punktach końcowych (jeden wiersz = jedna drużyna w sezonie).
5. Dla prognozy podajemy cechy **jednej drużyny** i otrzymujemy rozkład jej punktów (`predict_team_points`).
6. *(Opcjonalnie, `05_backtest_models_comparison`)* Łączymy predykcje 20 klubów w tabelę ligową przez sortowanie po punktach.

Wynik główny modelu: rozkład punktów dla danej drużyny (mediana, kwantyle, niepewność).

## Dane wejściowe

Źródłem danych są pliki `season-*.csv` z katalogu `Data/datahub.io`. Każdy wiersz to jeden mecz. Do modeli drużynowych używamy danych po agregacji do poziomu drużyna–sezon.

Najpierw funkcja `compute_table` buduje końcową tabelę sezonu. Następnie `load_season_tables(..., with_features=True)` dokleja cechy procesowe. Domyślnie używane są **lagowane cechy**, czyli dla targetu z sezonu `s` model dostaje informacje z sezonu `s-1`.

Jedna obserwacja w modelu oznacza:

```text
jedna drużyna w jednym sezonie -> końcowe punkty + cechy
```

Dla 16 sezonów treningowych po 20 drużyn mamy 320 obserwacji. Lagowanie cech jest ważne, bo zapobiega przeciekowi informacji: model nie widzi strzałów celnych ani formy z sezonu, którego punkty ma przewidzieć.

In [2]:
tables[["season", "team", "Pts", "sot_diff_pg", "pts_lag1", "ppg_last10"]].head(10)

,season,team,Pts,sot_diff_pg,pts_lag1,ppg_last10
0,0910,Chelsea,86,0.0,0.0,0.0
1,0910,Man United,85,0.0,0.0,0.0
2,0910,Arsenal,75,0.0,0.0,0.0
3,0910,Tottenham,70,0.0,0.0,0.0
4,0910,Man City,67,0.0,0.0,0.0
5,0910,Aston Villa,64,0.0,0.0,0.0
6,0910,Liverpool,63,0.0,0.0,0.0
7,0910,Everton,61,0.0,0.0,0.0
8,0910,Birmingham,50,0.0,0.0,0.0
9,0910,Blackburn,50,0.0,0.0,0.0


## Cechy używane w modelach

Zmienna objaśniana:

- `Pts` - końcowa liczba punktów drużyny w sezonie.

Predyktory:

- `sot_diff_pg` - różnica strzałów celnych na mecz z poprzedniego sezonu,
- `pts_lag1` - punkty z poprzedniego sezonu; dla beniaminków i pierwszego sezonu w danych przyjmujemy 0,
- `ppg_last10` - punkty na mecz w ostatnich 10 kolejkach poprzedniego sezonu,
- `is_promoted` - 1, jeśli drużyna nie grała w poprzednim sezonie Premier League (beniaminek), inaczej 0.

Wszystkie trzy cechy ciągłe są standaryzowane do z-score w funkcji `_standardize_features`. Dzięki temu współczynniki regresji mówią o zmianie oczekiwanych punktów przy zmianie cechy o jedno odchylenie standardowe. Standaryzacja jest liczona tylko na danych treningowych, a potem te same średnie i odchylenia są używane dla forecastu.

In [3]:
tables[["Pts", "sot_diff_pg", "pts_lag1", "ppg_last10"]].describe().round(2)

,Pts,sot_diff_pg,pts_lag1,ppg_last10
count,320.00,320.00,320.00,320.00
mean,52.42,0.30,45.02,1.21
std,17.70,1.70,26.80,0.78
min,12.00,-3.76,0.00,0.00
25%,40.00,-0.74,38.00,0.70
50%,48.00,0.00,47.00,1.30
75%,65.25,1.14,63.25,1.80
max,100.00,6.03,100.00,3.00


## Dlaczego Student-t?

Oba modele drużynowe używają rozkładu Studenta:

$$Pts_n \sim t_{\nu}(\mu_n, \sigma_{pts})$$

W projekcie `nu = 5` jest stałe. Student-t jest bardziej odporny na nietypowe sezony niż rozkład normalny. To ma sens dla tabel ligowych, bo zdarzają się sezony odstające: bardzo mocny mistrz, wyjątkowo słaby spadkowicz, nietypowy sezon beniaminka albo drużyna z dużymi problemami kadrowymi.

`sigma_pts` nie oznacza surowego odchylenia standardowego punktów w tabeli. Oznacza skalę reszt po uwzględnieniu cech i skillu.

## Model 1: `team_static.stan`

Model statyczny zakłada, że każda drużyna ma jeden latentny skill w całym okresie treningowym.

$$Pts_{s,t} \sim t_{\nu}(\mu_{s,t}, \sigma_{pts})$$

$$\mu_{s,t} = \alpha + \beta_{pts} skill_t + \beta_{sot} x^{sot}_{s,t} + \beta_{lag} x^{lag}_{s,t} + \beta_{form} x^{form}_{s,t} + \beta_{promoted}\, is\_promoted_{s,t}$$

Do Stana wrzucamy:

- `N` - liczba obserwacji drużyna-sezon,
- `T` - liczba drużyn obecnych w treningu,
- `team[n]` - indeks drużyny,
- `pts[n]` - końcowe punkty,
- `sot_diff_pg[n]`, `pts_lag1[n]`, `ppg_last10[n]` - wystandaryzowane cechy,
- `is_promoted[n]` - 1 dla beniaminka / nowej drużyny w sezonie, inaczej 0,
- `nu` - stopnie swobody rozkładu Studenta.

Mapowanie cech: $x^{sot}$ = `sot_diff_pg`, $x^{lag}$ = `pts_lag1`, $x^{form}$ = `ppg_last10`.

Ten model jest prosty i stabilny. Dobrze działa, jeśli stała jakość drużyny jest silnym sygnałem, a różnice między sezonami da się częściowo opisać cechami.

In [4]:
static_data, static_team_to_idx, static_teams, static_feature_stats = prepare_table_stan_static(
    tables,
    BACKTEST_TRAIN_SEASONS,
)

{k: (len(v) if hasattr(v, "__len__") and k not in {"N", "T", "nu"} else v) for k, v in static_data.items()}

{'N': 320,
 'T': 42,
 'nu': 5.0,
 'team': 320,
 'pts': 320,
 'sot_diff_pg': 320,
 'pts_lag1': 320,
 'ppg_last10': 320,
 'is_promoted': 320}

### Priory w `team_static.stan`

```stan
intercept ~ normal(52, 15);
beta_pts ~ normal(10, 10);
beta_sot ~ normal(0, 10);
beta_lag ~ normal(0, 1);
beta_form ~ normal(0, 10);
beta_promoted ~ normal(-8, 8);
log_sigma_pts ~ normal(log(15), 0.6);
skill ~ std_normal();
```

`intercept ~ normal(52, 15)` reprezentuje przeciętną drużynę Premier League. Środek prioru wynika ze struktury ligi 20-zespołowej, a szerokie odchylenie zostawia dużą niepewność.

`skill ~ std_normal()` nadaje drużynom latentną jakość na skali standardowej. W Stan używamy `sum_to_zero_vector[T]`, więc średnia skillu jest równa zero i model jest identyfikowalny bez dodatkowego centrowania.

`beta_pts ~ normal(10, 10)` przeskalowuje latentny skill na punkty. Prior jest słabo informatywny i dopuszcza zarówno mały, jak i duży wpływ latentnej jakości.

`beta_sot ~ normal(0, 10)` i `beta_form ~ normal(0, 10)` są szerokie, bo cechy są standaryzowane. Prior nie zakłada z góry kierunku ani konkretnej siły efektu.

`beta_lag ~ normal(0, 1)` pozwala poprzedniemu sezonowi pomagać w predykcji, ale nie wymusza silnego efektu.

`beta_promoted ~ normal(-8, 8)` dodaje słabe przekonanie, że beniaminkowie mogą być poniżej średniej, ale zostawia dużą niepewność.

`log_sigma_pts ~ normal(log(15), 0.6)` daje szeroki prior na skalę reszt. Nie jest ustawiany tak, żeby dopasować historyczny histogram punktów; ma tylko utrzymać dodatnią, piłkowo sensowną skalę.

## Model 2: `team_hierarchical.stan`

Model hierarchiczny jest rozszerzeniem modelu drużynowego. Obecna wersja rozdziela skill na dwie części:

$$skill_{s,t} = team\_skill_t + season\_dev_{s,t}$$

- `team_skill[t]` - trwała jakość drużyny,
- `season_effect[s]` - wspólne odchylenie poziomu punktów w danym sezonie.
- brak osobnego `season_dev[s,t]`; indywidualne niespodzianki drużyna-sezon zostają w residual Student-t.

Równanie modelu:

$$Pts_{s,t} \sim t_{\nu}(\mu_{s,t}, \sigma_{pts})$$

$$\mu_{s,t} = \alpha + team\_skill_t + season\_dev_{s,t} + \beta_{sot} x^{sot}_{s,t} + \beta_{lag} x^{lag}_{s,t} + \beta_{form} x^{form}_{s,t} + \beta_{promoted}\, is\_promoted_{s,t}$$

To jest kompromis między modelem statycznym a modelem w pełni sezonowym. Model zachowuje informację, że drużyny mają trwałą jakość, ale pozwala też na sezonowe zmiany.

In [5]:
hier_data, hier_team_to_idx, hier_teams, hier_season_to_idx, hier_feature_stats = prepare_table_stan_hierarchical(
    tables,
    BACKTEST_TRAIN_SEASONS,
)

{k: (len(v) if hasattr(v, "__len__") and k not in {"N", "S", "T", "nu"} else v) for k, v in hier_data.items()}

{'N': 320,
 'S': 16,
 'T': 42,
 'nu': 5.0,
 'season': 320,
 'team': 320,
 'pts': 320,
 'sot_diff_pg': 320,
 'pts_lag1': 320,
 'ppg_last10': 320,
 'is_promoted': 320}

### Co dodatkowo trafia do modelu hierarchicznego?

Poza wspólnymi danymi z modelu statycznym (`pts`, cechy, `is_promoted`, `nu`) model hierarchiczny dostaje:

- `S` - liczba sezonów,
- `season[n]` - indeks sezonu obserwacji,
- `team[n]` - indeks drużyny obserwacji (jak w Model 1).

Sezonowe odchylenia są zapisane jako `array[S] sum_to_zero_vector[T]`. Predykcja dotyczy **jednej drużyny** — jej cech + skill → rozkład punktów.

### Priory w `team_hierarchical.stan`

```stan
intercept ~ normal(52, 15);
beta_sot ~ normal(0, 10);
beta_lag ~ normal(0, 1);
beta_form ~ normal(0, 10);
beta_promoted ~ normal(-8, 8);
log_sigma_pts ~ normal(log(8), 0.5);
log_tau_team ~ normal(log(8), 0.5);
log_tau_season ~ normal(log(3), 0.5);
team_skill_z ~ std_normal();
season_effect_z ~ std_normal();
```

`log_sigma_pts ~ normal(log(8), 0.5)` opisuje resztkowy szum punktowy po uwzględnieniu cech, trwałej jakości drużyny i sezonowego odchylenia. Prior jest szeroki, żeby nie wymuszać konkretnej skali reszt przed zobaczeniem danych.

`log_tau_team ~ normal(log(8), 0.5)` opisuje rozrzut trwałej jakości drużyn. To luźne przypuszczenie, że persistent quality może mieć kilka do kilkunastu punktów wpływu.

`log_tau_season ~ normal(log(3), 0.5)` opisuje wspólne przesunięcia sezonowe. Każdy sezon ma około 20 drużyn, więc taki efekt jest dużo lepiej identyfikowalny niż osobny efekt dla każdej pary drużyna-sezon.

Wcześniejsza wersja zawierała osobny sezonowy efekt dla każdej pary drużyna-sezon. Został usunięty, ponieważ pojedynczy team-season ma tylko jedną obserwację punktów, więc ten efekt był słabo identyfikowalny i konkurował z `sigma_pts`.

`beta_promoted ~ normal(-8, 8)` działa tak samo jak w modelu statycznym: daje słabe oczekiwanie niższych punktów dla beniaminków, z dużą niepewnością.

Parametry są zapisane na log-skali, żeby skale były dodatnie. Dodatkowo mają szerokie ograniczenia domenowe, np. `log_sigma_pts` między `log(0.5)` i `log(20)`, co poprawia stabilność numeryczną inicjalizacji.

## Dlaczego model hierarchiczny nie powinien być zbyt elastyczny?

Model drużynowy ma mało obserwacji: jedna liczba punktów na drużynę w sezonie. Jeśli każdej parze sezon–drużyna damy prawie niezależny skill, model może bardzo dobrze dopasować historię, ale gorzej prognozować przyszłość.

Dlatego finalna konstrukcja jest taka:

```text
skill = trwała jakość drużyny + małe sezonowe odchylenie
```

Trwała jakość przenosi informację między sezonami. Wspólny efekt sezonu pozwala przesuwać cały sezon w górę lub w dół, a indywidualna zmienność drużyn pozostaje w Student-t residual noise.

## Backtest i finalny forecast

W podstawowym backteście trenujemy modele na sezonach do 2024/25 i przewidujemy **punkty każdej drużyny** w sezonie 2025/26. Szczegóły per model: `02_model1_static_team` (Model 1) i `03_model2_hierarchical_team` (Model 2).

Notebook `05_backtest_models_comparison` porównuje **oba modele** na tym samym backteście: błędy punktów per drużyna, suma błędów sezonu i średnia na drużynę (`sum |error| / n_teams`).

W finalnym forecastcie trenujemy na wszystkich dostępnych sezonach, włącznie z 2025/26, i przewidujemy sezon 2026/27 (`04_forecast_2627_comparison`).

Dla drużyn prognozowanego sezonu cechy są budowane na podstawie ostatniego dostępnego sezonu. Drużyny, które nie grały w poprzednim sezonie Premier League, dostają `is_promoted=1`, a modele mają parametr `beta_promoted`, który może obniżyć ich oczekiwane punkty.

**Skill w prognozie:** Model 1 używa stałego `skill[team]`. Model 2 bierze `team_skill[team]` z posterioru i losuje nowy wspólny `season_effect` dla prognozowanego sezonu. Drużyny bez historii w treningu dostają skill = 0; pozostała niepewność pochodzi z predykcyjnego Student-t residual noise oraz `beta_promoted` dla beniaminków.

## Diagnostyka modeli

Po dopasowaniu modelu sprawdzamy:

- `R_hat` - powinno być blisko 1.00, zwykle akceptujemy do 1.01,
- `ESS_bulk` i `ESS_tail` - im większe, tym lepiej; niskie ESS oznacza słabe mieszanie łańcuchów,
- divergences - powinno być 0,
- treedepth - nie powinien masowo dobijać do maksimum,
- E-BFMI - niskie wartości sugerują problemy z geometrią posterioru.

Diagnostyka samplera mówi, czy posterior został dobrze wysamplowany. Nie mówi jeszcze, czy forecast jest dobry. Jakość predykcji sprawdzamy osobno przez backtest: błędy punktów (`predict_team_points` vs rzeczywiste Pts) w `02_model1_static_team`, `03_model2_hierarchical_team` i porównanie obu modeli w `05_backtest_models_comparison`.

## Krótkie porównanie modeli drużynowych

`team_static`:

- prostszy,
- stabilny diagnostycznie,
- ma jeden skill drużyny dla wszystkich sezonów,
- często daje dobrą predykcję, bo mocno korzysta z trwałej jakości drużyn.

`team_hierarchical`:

- bardziej elastyczny,
- rozdziela trwały skill drużyny i sezonowe odchylenie,
- wymaga mocniejszej regularizacji,
- może lepiej uchwycić zmiany formy, ale tylko jeśli sezonowe odchylenia nie są zbyt szerokie.

W praktyce model hierarchiczny nie powinien być oceniany po samej złożoności. Trzeba patrzeć na backtest. Bardziej złożony model może gorzej przewidywać, jeśli dodatkowa elastyczność łapie szum historyczny.